# Stockholm Bus Deadhead Analysis

This notebook fetches GTFS-RT vehicle position data from the Trafiklab KoDa API and analyzes deadhead (tomkörning) patterns for Stockholm bus routes.

**Works in:** Google Colab, GitHub Codespaces, or any local Jupyter environment.

## 1. Environment Setup

Installs dependencies and sets up the project path. Handles both Colab and Codespaces automatically.

In [ ]:
import os
import sys

# Detect environment
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Clone the repo into Colab (or pull latest if already cloned)
    # Set GITHUB_TOKEN to enable push: os.environ["GITHUB_TOKEN"] = "ghp_..."
    REPO_BRANCH = "claude/fix-data-analysis-ihWlO"
    GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "")
    if GITHUB_TOKEN:
        REPO_URL = f"https://{GITHUB_TOKEN}@github.com/HEVI-SE/Trafiklab.git"
    else:
        REPO_URL = "https://github.com/HEVI-SE/Trafiklab.git"
    REPO_DIR = "/content/Trafiklab"
    if not os.path.exists(REPO_DIR):
        !git clone -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}
    else:
        !cd {REPO_DIR} && git fetch origin {REPO_BRANCH} && git checkout {REPO_BRANCH} && git pull origin {REPO_BRANCH}
    os.chdir(REPO_DIR)
    sys.path.insert(0, REPO_DIR)
    # Re-set remote URL with token so push works
    if GITHUB_TOKEN:
        !cd {REPO_DIR} && git remote set-url origin {REPO_URL}
    print(f"Colab: working directory set to {REPO_DIR}")
    if not GITHUB_TOKEN:
        print("⚠ GITHUB_TOKEN ej satt — git push kommer inte fungera. Sätt os.environ['GITHUB_TOKEN'] = 'ghp_...'")
else:
    # Codespaces / local: ensure we're in the repo root
    notebook_dir = os.path.dirname(os.path.abspath("__file__"))
    if os.path.exists(os.path.join(notebook_dir, "config.py")):
        os.chdir(notebook_dir)
        sys.path.insert(0, notebook_dir)
    print(f"Local: working directory is {os.getcwd()}")

# Install dependencies
!pip install -q pandas requests py7zr plotly gtfs-realtime-bindings

print("Setup complete!")

## 2. Configuration

Set the date and hours you want to analyze. The API key has a default but you can override it.

In [ ]:
# ---- EDIT THESE ----
# Months to fetch: April, May, Sep, Oct, Nov 2025 and Feb, Mar 2026
MONTH_RANGES = [
    ("2025-04-01", "2025-04-30"),
    ("2025-05-01", "2025-05-31"),
    ("2025-09-01", "2025-09-30"),
    ("2025-10-01", "2025-10-31"),
    ("2025-11-01", "2025-11-30"),
    ("2026-02-01", "2026-02-28"),
    ("2026-03-01", "2026-03-28"),
]

# Full 24h fetch (all hours)
HOURS = list(range(0, 24))

# Optional: override the default API key
os.environ["KODA_API_KEY"] = "4psdkvdO9UIYsziDkp3AlnGUL5N5a4tE19N2TSja28I"

# Generate date list from all month ranges
from datetime import datetime, timedelta
DATES = []
for start_str, end_str in MONTH_RANGES:
    _start = datetime.strptime(start_str, "%Y-%m-%d")
    _end = datetime.strptime(end_str, "%Y-%m-%d")
    while _start <= _end:
        DATES.append(_start.strftime("%Y-%m-%d"))
        _start += timedelta(days=1)

print(f"Will analyze: {len(DATES)} day(s) across {len(MONTH_RANGES)} month ranges")
print(f"Periods: Apr 2025, May 2025, Sep-Nov 2025, Feb-Mar 2026")

## 3. Load Static GTFS Schedule

Downloads the static GTFS data for **each month range** and merges them. This ensures stops and trips from all periods are available for analysis.

In [ ]:
import pandas as pd
from config import OPERATOR_MAPPING
from fetcher import load_static_gtfs, build_trip_lookup

# Load GTFS for one representative date per month range and merge
# This ensures stops/trips from ALL periods are available
_gtfs_dates = sorted(set(start for start, _ in MONTH_RANGES))
print(f"Laddar GTFS för {len(_gtfs_dates)} perioder: {', '.join(_gtfs_dates)}")

_all_routes, _all_trips, _all_stops, _all_stop_times = [], [], [], []
for _date in _gtfs_dates:
    try:
        r, t, s, st = load_static_gtfs(_date)
        _all_routes.append(r)
        _all_trips.append(t)
        _all_stops.append(s)
        _all_stop_times.append(st)
        print(f"  {_date}: {len(r)} routes, {len(t)} trips, {len(s)} stops")
    except Exception as e:
        print(f"  {_date}: FEL - {e}")

# Merge and deduplicate
routes = pd.concat(_all_routes, ignore_index=True).drop_duplicates(subset=["route_id"], keep="last")
trips = pd.concat(_all_trips, ignore_index=True).drop_duplicates(subset=["trip_id"], keep="last")
stops = pd.concat(_all_stops, ignore_index=True).drop_duplicates(subset=["stop_id"], keep="last")
stop_times = pd.concat(_all_stop_times, ignore_index=True).drop_duplicates(
    subset=["trip_id", "stop_id", "stop_sequence"], keep="last"
)

print(f"\nMerged: Routes: {len(routes)}, Trips: {len(trips)}, Stops: {len(stops)}, Stop times: {len(stop_times)}")

## 4. Build Trip Lookup Table

Creates a lookup mapping trip_id to route, operator, headsign, first/last stop.

In [ ]:
operator_df = pd.DataFrame(OPERATOR_MAPPING)
trip_lookup = build_trip_lookup(trips, routes, operator_df, stop_times, stops)

print(f"Trip lookup: {len(trip_lookup)} trips")
print(f"Operators: {trip_lookup['operator'].value_counts().to_dict()}")
trip_lookup.head()

## 5. Fetch Vehicle Positions

Downloads GTFS-RT vehicle positions for full 24h days. Already-fetched days (tracked in `data/fetched_days.csv`) are skipped automatically.

**First run** for a new date takes 15-30 minutes. Subsequent runs with the same dates are instant.

In [ ]:
import importlib
from fetcher import fetch_vehicle_positions, filter_bus_segments
import csv_handler as _csv_mod
importlib.reload(_csv_mod)
from csv_handler import load_segments, get_fetched_days, mark_day_fetched

# Check which days are already fully fetched (tracked in git)
fetched_days = get_fetched_days()
days_to_fetch = [d for d in DATES if d not in fetched_days]
days_cached = [d for d in DATES if d in fetched_days]

if days_cached:
    print(f"Redan hämtade dagar (hoppar över): {', '.join(days_cached)}")
if days_to_fetch:
    print(f"Dagar att hämta: {', '.join(days_to_fetch)}")
else:
    print("Alla dagar redan hämtade!")

# Load ALL cached segments (includes all previously fetched days)
cached_segments = load_segments()

# Fetch only missing days (full 24h each)
all_segments = []
for date in days_to_fetch:
    print(f"\n=== {date} (24 timmar) ===")
    seg = fetch_vehicle_positions(date, HOURS, trip_lookup)
    if not seg.empty:
        all_segments.append(seg)
        mark_day_fetched(date)
    else:
        print(f"  Varning: ingen data för {date}")

new_segments = pd.concat(all_segments, ignore_index=True) if all_segments else pd.DataFrame()

# Combine ALL cached + new (all historical data included)
if not cached_segments.empty and not new_segments.empty:
    segments = pd.concat([cached_segments, new_segments], ignore_index=True)
elif not cached_segments.empty:
    segments = cached_segments
else:
    segments = new_segments

# Filter to bus vehicles only
segments = filter_bus_segments(segments)

# Deduplicate
dedup_cols = ["vehicle_id", "start_time", "end_time", "route_short_name"]
available = [c for c in dedup_cols if c in segments.columns]
segments = segments.drop_duplicates(subset=available, keep="last").reset_index(drop=True)

# Show stats for ALL data
if not segments.empty:
    seg_dates = pd.to_datetime(segments["start_time"]).dt.date.unique()
    print(f"\nTotal segment (bussar, {len(seg_dates)} dagar): {len(segments):,}")
    print(f"Unika fordon: {segments['vehicle_id'].nunique()}")
    print(f"Observerade linjer: {segments['route_short_name'].nunique()}")
    print(f"Datumintervall: {min(seg_dates)} – {max(seg_dates)}")
else:
    print("\nInga segment laddade.")
segments.head(10)

## 6. Detect Deadheads (Tomkörningar)

Identifies periods where buses travel empty between trips.

In [ ]:
import importlib
import analysis as _analysis_mod
importlib.reload(_analysis_mod)
from analysis import build_observed_deadheads, build_planned_deadheads, filter_deadheads_osrm, build_dwell_lookup

# Observed deadheads (from real-time vehicle tracking)
observed = build_observed_deadheads(segments, stops)
print(f"Observed deadheads (raw): {len(observed)}")

# Planned deadheads (from static GTFS schedule)
planned = build_planned_deadheads(trips, stop_times, stops, routes, operator_df)
print(f"Planned deadheads (raw): {len(planned)}")

# OSRM filter: remove deadheads with unrealistic durations
# (>100% slower or >50% faster than OSRM driving estimate)
if not observed.empty:
    observed = filter_deadheads_osrm(observed, min_ratio=0.5, max_ratio=2.0)
    print(f"Observed after OSRM filter: {len(observed)}")

# For planned: subtract observed dwell time per stop pair before OSRM comparison
# (planned duration includes idle time at terminals that observed data reveals)
if not planned.empty:
    dwell = build_dwell_lookup(observed)
    print(f"Dötidslookup: {len(dwell)} hållplatspar med observerad dötid")
    planned = filter_deadheads_osrm(planned, min_ratio=0.5, max_ratio=2.0, dwell_lookup=dwell)
    print(f"Planned after OSRM filter: {len(planned)}")

## 7. Save Results to CSV

Saves segments and deadheads to CSV files in the `data/` directory. Automatically deduplicates with any existing data.

In [ ]:
importlib.reload(_csv_mod)
from csv_handler import save_segments, save_deadheads, push_data_to_git

save_segments(segments)

if not observed.empty:
    save_deadheads(observed)

if not planned.empty:
    save_deadheads(planned)

print("\nResults saved to data/ directory.")

# Push data to git so next run can reuse it
date_label_csv = f"{DATES[0]} to {DATES[-1]}" if len(DATES) > 1 else DATES[0]
push_data_to_git(f"Cache data for {date_label_csv}")

## 8. Analysis Summary

Overview statistics of the collected data.

In [ ]:
from utils import classify_period

# Derive stats from actual data (all fetched days, not just DATES)
print("=" * 60)
print("ANALYS — ALL HÄMTAD DATA")
print("=" * 60)

n_obs = len(observed) if not observed.empty else 0
n_unique = observed.drop_duplicates(subset=["from_stop_observed", "to_stop_observed"]).shape[0] if n_obs > 0 else 0

# Get actual date range from data
if n_obs > 0:
    obs_dates = pd.to_datetime(observed["deadhead_start"]).dt.date
    all_data_dates = sorted(obs_dates.unique())
    n_days = len(all_data_dates)
    date_range_str = f"{all_data_dates[0]} – {all_data_dates[-1]}"
    ALL_DATA_DATES = [str(d) for d in all_data_dates]
else:
    n_days = len(DATES)
    date_range_str = f"{DATES[0]} – {DATES[-1]}"
    ALL_DATA_DATES = DATES

print(f"\nObserverade tomkörningar: {n_obs:,}")
print(f"Unika hållplatspar: {n_unique:,}")
print(f"Analyserade dagar: {n_days}")
print(f"Analysperiod: {date_range_str}")

if n_obs > 0:
    print(f"\nSnitt restid: {observed['duration_min'].mean():.1f} min")
    if 'beräknad_körtid_min' in observed.columns:
        osrm_mean = observed['beräknad_körtid_min'].mean()
        print(f"Snitt beräknad tid (OSRM): {osrm_mean:.1f} min")
    print(f"\nPer dagtyp:")
    if 'day_type' in observed.columns:
        print(observed['day_type'].value_counts().to_string())
    print(f"\nPer trafikperiod:")
    print(observed['period'].value_counts().to_string())
    print(f"\nPer operatör:")
    print(observed['operator'].value_counts().to_string())

## 9. Visualizations

In [ ]:
import plotly.express as px

# Use actual data date range for chart labels
chart_dates = ALL_DATA_DATES if 'ALL_DATA_DATES' in dir() else DATES
date_label = f"{chart_dates[0]} to {chart_dates[-1]}" if len(chart_dates) > 1 else chart_dates[0]

# Deadheads by operator
if not observed.empty:
    fig = px.histogram(
        observed, x="operator", color="period",
        title=f"Observerade tomkörningar per operatör & trafikperiod ({date_label})",
        labels={"operator": "Operatör", "count": "Antal", "period": "Trafikperiod"},
        barmode="stack",
    )
    fig.update_layout(xaxis_categoryorder="total descending")
    fig.show()

In [ ]:
# Deadhead duration distribution
if not observed.empty:
    fig = px.histogram(
        observed, x="duration_min", nbins=30,
        title=f"Deadhead Duration Distribution ({date_label})",
        labels={"duration_min": "Duration (minutes)", "count": "Count"},
    )
    fig.show()

In [ ]:
# Top deadhead stop pairs
if not observed.empty:
    top_stops = (
        observed.groupby(["from_stop_observed", "to_stop_observed"])
        .agg(count=("vehicle_id", "size"), avg_duration=("duration_min", "mean"), avg_distance=("move_m", "mean"))
        .reset_index()
        .sort_values("count", ascending=False)
        .head(15)
    )
    top_stops["stop_pair"] = top_stops["from_stop_observed"] + " → " + top_stops["to_stop_observed"]
    top_stops["avg_duration"] = top_stops["avg_duration"].round(1)
    top_stops["avg_distance"] = (top_stops["avg_distance"] / 1000).round(1)

    fig = px.bar(
        top_stops, x="stop_pair", y="count",
        hover_data=["avg_duration", "avg_distance"],
        title=f"Top 15 Deadhead Stop Pairs ({date_label})",
        labels={"stop_pair": "Hållplatspar", "count": "Antal", "avg_duration": "Snitt min", "avg_distance": "Snitt km"},
    )
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()

In [ ]:
# Deadhead timeline
if not observed.empty:
    timeline = observed.copy()
    timeline["hour"] = pd.to_datetime(timeline["deadhead_start"]).dt.hour
    hourly = timeline.groupby("hour").agg(
        count=("vehicle_id", "size"),
        total_km=("move_m", lambda x: (x.sum() / 1000).round(1)),
    ).reset_index()

    fig = px.bar(
        hourly, x="hour", y="count",
        hover_data=["total_km"],
        title=f"Deadheads by Hour of Day ({date_label})",
        labels={"hour": "Hour", "count": "Number of Deadheads", "total_km": "Total Distance (km)"},
    )
    fig.update_layout(xaxis_dtick=1)
    fig.show()

## 10. Fetch Delay Data & Build Line Stop Data

Fetches GTFS-RT TripUpdates for per-stop delay analysis and builds the line stop sequences for the map view.

In [ ]:
from fetcher import fetch_trip_updates, build_line_stop_data

# Fetch TripUpdates for ALL days with data (not just configured DATES)
# ALL_DATA_DATES is set in the summary cell above and includes every date in the dataset
delay_dates = ALL_DATA_DATES if 'ALL_DATA_DATES' in dir() else DATES

print(f"Hämtar förseningsdata för {len(delay_dates)} dag(ar): {', '.join(delay_dates)}")

all_delays = []
for date in delay_dates:
    try:
        print(f"\n=== TripUpdates {date} ===")
        d = fetch_trip_updates(date, HOURS, trip_lookup)
        if not d.empty:
            all_delays.append(d)
            print(f"  {len(d):,} poster")
    except Exception as e:
        print(f"  Kunde inte hämta TripUpdates för {date}: {e}")

if all_delays:
    delays_df = pd.concat(all_delays, ignore_index=True)
    print(f"\nTotal förseningsdata: {len(delays_df):,} poster, {delays_df['route_short_name'].nunique()} linjer")
else:
    delays_df = None
    print("\nIngen förseningsdata tillgänglig.")

# Build line stop data for map view
line_stop_data = build_line_stop_data(routes, trips, stop_times, stops, delays_df)
print(f"Linjedata: {len(line_stop_data)} linje-riktningar")

## 11. Generate HTML Report

Ljust tema med gul accent. Inkluderar all hämtad data:
- **Linjer**: Välj busslinje och se försening per hållplats på karta
- **Tomkörningar**: Tomkörningar grupperade per hållplatspar med period- och dagtypfilter

In [ ]:
import importlib
import report as _report_mod
importlib.reload(_report_mod)
from report import generate_html_report

# Use actual data dates for report (ALL_DATA_DATES includes all fetched days)
report_dates = ALL_DATA_DATES if 'ALL_DATA_DATES' in dir() else DATES
date_label = f"{report_dates[0]}_to_{report_dates[-1]}" if len(report_dates) > 1 else report_dates[0]

report_path = generate_html_report(
    observed, planned, segments, date_label,
    line_stop_data=line_stop_data,
    dates=report_dates, hours=HOURS,
)

# Auto-download in Colab, or print path for local use
if IN_COLAB:
    from google.colab import files
    files.download(report_path)
    print("Download started!")
else:
    import webbrowser
    abs_path = os.path.abspath(report_path)
    print(f"Rapport sparad: {abs_path}")
    try:
        webbrowser.open(f"file://{abs_path}")
        print("Öppnad i webbläsare.")
    except Exception:
        print("Öppna filen ovan i din webbläsare.")

## 12. Explore Raw Data

Browse the raw dataframes interactively.

In [ ]:
# View observed deadheads
if not observed.empty:
    display_cols = ["operator", "prev_route", "next_route", "from_stop_observed", "to_stop_observed",
                    "deadhead_start", "duration_min", "beräknad_körtid_min", "move_m", "speed_kmh", "period", "day_type"]
    available = [c for c in display_cols if c in observed.columns]
    observed[available].head(20)
else:
    print("No observed deadheads to show.")